# RegMixMatch

RegMixMatch (Tao et al.), ported from the reference implementation
(https://github.com/hhrd9/regmixmatch), itself built on FreeMatch (Wang et al., 2023). It has
three ingredients:

- **FreeMatch's self-adaptive threshold** replaces a fixed $\tau$: a global level `time_p` and
  a per-class modulation `p_model`, both moving averages of the model's own predictions on
  the weak view;
- **FreeMatch's fairness term** (`entropy_loss`) counteracts pseudo-labels collapsing onto a
  few classes;
- **RegMixMatch's own contribution**, a confidence-routed ResizeMix: the images whose
  confidence exceeds $\tau_m = 0.999$ (plus the labeled batch when labels are plentiful) form a
  *confident pool*, which is mixed with itself to produce extra supervised signal (`mix_loss`).

$$\mathcal{L} = \mathcal{L}_s + \lambda_e\,\mathcal{L}_{ent} + \lambda_u\,\mathcal{L}_u + \mathcal{L}_{mix}$$

**Departures from the reference implementation.**

- The reference code rebuilds the confident pool by slicing the batch, so its size changes at
  every iteration. The option `STATIC_SHAPES` instead mixes the whole batch and zeroes the
  non-confident contributions with a mask. What is optimised is the same; only the images that
  can donate a ResizeMix patch differ. Fixed shapes are what `torch.compile`'s CUDA graphs
  require. The main runs of the paper use the slicing path (`STATIC_SHAPES = False`), and the
  matched-mu runs of Table 10 use the mask.
- The threshold's moving averages are seeded from the unlabeled set, not from the test set as
  in the reference code: training never touches test data.
- The second, class-aware ResizeMix branch of RegMixMatch (`cam_loss`) is left out of this
  notebook. The upstream authors recommend disabling it, and no run of the paper uses it; it
  remains in `scripts/regmixmatch.py` (`--disab_cam False`).

## How to use this notebook

Run the cells from top to bottom. With `QUICK_TEST = True`, the default, the run stops after
1,000 iterations: enough to check that everything works and to watch the first evaluations, in a
few minutes on a GPU. Set it to `False` for a run of the paper, which stops at the target
accuracy or after two hours.

Everything the method does is written out below: the labeled/unlabeled split, the
augmentations, the EMA, the losses, the training loop and the evaluation. Only the WideResNet
architecture is imported, from `scripts/models.py`, since it is the same for every method. Edit
any cell, then re-run it and every cell after it.

The run is written to `results/<dataset>-labeled-<n>-seed-<seed>/regmixmatch_ema_notebook_metrics.json`,
next to the paper's `regmixmatch_ema_metrics.json` and never over it. `figures.ipynb` and
`python scripts/run_analysis.py compare` both read it.

At the default settings this notebook is the same computation as `scripts/regmixmatch.py`: same
labeled subset, same initial weights, same random draws. With
`torch.backends.cudnn.benchmark = False` in both, the first iteration gives bit-for-bit the
loss the script gives. With it on, as in the paper, cuDNN picks its convolution algorithms by
timing them and may not pick the same ones here as in the script, which moves the last
digits. After the first iteration, two runs drift apart slightly in any case — two runs of the
script do too — because bfloat16, TF32 and cuDNN are not bit-reproducible
(see `results/README.md`). For RegMixMatch, the 2,048 warmup iterations of section 5 come first and already drift from one run to the next; with `WARMUP_STEPS` set to a few iterations, the first training iteration matches the script's.

In [ ]:
import json
import math
import sys
import time
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from sklearn.metrics import accuracy_score, f1_score
from torch.amp import autocast
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import v2

# The repository root is the folder holding run_experiment.py, wherever Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_experiment.py").exists())
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from models import build_model   # WideResNet-28-k, the same for every method

## 1. Configuration

Every setting lives in the next three cells: the experiment, the method's own hyperparameters,
and the settings shared by all methods. The values given are the paper's.

In [ ]:
# --- Experiment ---------------------------------------------------------------------------------
# The four configurations of the paper. Pick one with CONFIG, or overwrite the values below it.
PAPER_CONFIGS = {
    "svhn-250":      dict(dataset="svhn",     num_labeled=250,  target_acc=0.90, widen_factor=2),
    "cifar10-250":   dict(dataset="cifar10",  num_labeled=250,  target_acc=0.80, widen_factor=2),
    "cifar10-4000":  dict(dataset="cifar10",  num_labeled=4000, target_acc=0.90, widen_factor=2),
    "cifar100-2500": dict(dataset="cifar100", num_labeled=2500, target_acc=0.50, widen_factor=4),
}
CONFIG = "cifar10-250"
SEED = 2312                  # the paper reports seeds 2312, 308 and 2701

DATASET      = PAPER_CONFIGS[CONFIG]["dataset"]
NUM_LABELED  = PAPER_CONFIGS[CONFIG]["num_labeled"]    # split evenly across the classes
TARGET_ACC   = PAPER_CONFIGS[CONFIG]["target_acc"]     # stop as soon as the EMA model reaches it
WIDEN_FACTOR = PAPER_CONFIGS[CONFIG]["widen_factor"]   # WideResNet-28-2, or 28-4 on CIFAR-100
MAX_MINUTES  = 120                                     # the paper's two-hour budget

# --- Run length ---------------------------------------------------------------------------------
QUICK_TEST  = True           # False for a full run of the paper
MAX_STEPS   = 1_000 if QUICK_TEST else 2**20
TEST_PERIOD = 100 if QUICK_TEST else 500    # evaluate every N iterations; the paper uses 500
RUN_TAG     = "notebook"     # appended to the result file name, so no paper result is overwritten

In [ ]:
# --- RegMixMatch --------------------------------------------------------------------------------
MU            = 7        # unlabeled:labeled batch ratio (Table 10 also uses 3)
TAU_M         = 0.999    # confidence above which an unlabeled image joins the confident pool
LAMBDA_U      = 1.0      # weight of the consistency loss
LAMBDA_E      = 0.01     # weight of the fairness (entropy) loss
HARD_LABEL    = True     # argmax pseudo-label, or else a soft target sharpened by T
T             = 0.5      # sharpening temperature, only used when HARD_LABEL is False
SVHN_CLAMP    = True     # clamp the adaptive threshold to [0.9, 0.95] on SVHN, as the reference code does
ALPHA_H       = 1.0      # Beta concentration for the size of the ResizeMix box
WARMUP_STEPS  = 2048     # supervised-only iterations before training, to seed the threshold (section 5)
STATIC_SHAPES = False    # mask instead of slicing (see the introduction); needed for torch.compile

In [ ]:
# --- Optimisation: identical for every method of the paper ------------------------------------
BATCH_SIZE_L = 64            # labeled images per iteration; the unlabeled batch is MU times larger
LR           = 0.03
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
TOTAL_STEPS  = 2**20         # horizon of the cosine learning-rate decay, whatever MAX_STEPS is
EMA_DECAY    = 0.999         # the evaluated model is this moving average of the weights

# --- Runtime: these change the speed, not the algorithm -----------------------------------------
USE_AMP            = True    # bfloat16 autocast on the forward passes
CHANNELS_LAST      = True    # a memory layout that speeds up convolutions on recent GPUs
COMPILE            = STATIC_SHAPES   # CUDA graphs need fixed shapes (see the introduction)
NUM_WORKERS        = 2       # data-loading processes
PER_SAMPLE_AUGMENT = False   # see section 3; False is what the paper did

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (training will be very slow)")
torch.backends.cudnn.benchmark = True          # let cuDNN pick the fastest convolution algorithm
torch.set_float32_matmul_precision("high")     # TF32 matrix products

MEMORY_FORMAT = torch.channels_last if CHANNELS_LAST else torch.contiguous_format


def amp():
    """bfloat16 autocast for a forward pass, or nothing when it is turned off."""
    if USE_AMP and DEVICE.type == "cuda":
        return autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

## 2. Data

The labeled subset is drawn as in the scripts: `torch.manual_seed(SEED)`, one random
permutation of the whole training set, then `NUM_LABELED / num_classes` images per class; all
the others form the unlabeled set. Their labels are never used for training, only to score the
pseudo-labels.

The order of these draws matters. The model's initial weights are drawn next, from what is
left of the same random stream, so drawing anything else in between changes both the split and
the initialisation.

The images stay in memory as `uint8` tensors. Each batch is converted to floats and augmented
inside the training loop.

In [ ]:
DATA_ROOT = REPO_ROOT / "data"
DATASET_STATS = {   # number of classes, per-channel mean and standard deviation
    "cifar10":  (10,  (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    "cifar100": (100, (0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
    "svhn":     (10,  (0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970)),
}
NUM_CLASSES, MEAN, STD = DATASET_STATS[DATASET]


def load_split(name, train):
    """All the images of one split as a uint8 tensor (N, 3, 32, 32), with their labels.
    Downloads the dataset into data/ on first use."""
    if name == "svhn":
        ds = torchvision.datasets.SVHN(DATA_ROOT, split="train" if train else "test", download=True)
        return torch.from_numpy(ds.data), torch.from_numpy(ds.labels).long()
    cls = torchvision.datasets.CIFAR10 if name == "cifar10" else torchvision.datasets.CIFAR100
    ds = cls(DATA_ROOT, train=train, download=True)
    return torch.from_numpy(ds.data).permute(0, 3, 1, 2).contiguous(), torch.tensor(ds.targets)


def split_labeled_unlabeled(labels, num_classes, num_labeled):
    """Indices of the labeled images, num_labeled // num_classes per class drawn at random, and of
    the unlabeled ones (all the others)."""
    order = torch.randperm(len(labels))          # shuffle the whole training set once...
    per_class = num_labeled // num_classes
    labeled, unlabeled = [], []
    for c in range(num_classes):
        members = order[(labels[order] == c).nonzero(as_tuple=True)[0]]   # class c, shuffled order
        pick = torch.randperm(len(members))       # ...then draw within each class
        labeled.append(members[pick[:per_class]])
        unlabeled.append(members[pick[per_class:]])
    return torch.cat(labeled), torch.cat(unlabeled)


x_train, y_train = load_split(DATASET, train=True)
x_test, y_test = load_split(DATASET, train=False)

torch.manual_seed(SEED)
labeled_idx, unlabeled_idx = split_labeled_unlabeled(y_train, NUM_CLASSES, NUM_LABELED)
print(f"{len(labeled_idx)} labeled, {len(unlabeled_idx)} unlabeled, {len(y_test)} test images")
print("Labeled images per class:", torch.bincount(y_train[labeled_idx], minlength=NUM_CLASSES).tolist())

In [ ]:
labeled_set = TensorDataset(x_train[labeled_idx], y_train[labeled_idx])
# The unlabeled set also yields each image's position, to follow its pseudo-label over training.
unlabeled_set = TensorDataset(x_train[unlabeled_idx], y_train[unlabeled_idx], torch.arange(len(unlabeled_idx)))
test_set = TensorDataset(x_test, y_test)
unlabeled_true_labels = y_train[unlabeled_idx]   # only used to score the pseudo-labels

loader_kwargs = (dict(num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=4, persistent_workers=True)
                 if NUM_WORKERS > 0 else {})
labeled_loader = DataLoader(labeled_set, batch_size=BATCH_SIZE_L, shuffle=True, **loader_kwargs)
unlabeled_loader = DataLoader(unlabeled_set, batch_size=BATCH_SIZE_L * MU, shuffle=True, **loader_kwargs)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, **loader_kwargs)


def next_batch(iterator, loader):
    """The next batch, starting a new pass over the data when the loader is exhausted."""
    try:
        return next(iterator), iterator
    except StopIteration:
        iterator = iter(loader)
        return next(iterator), iterator

## 3. Augmentations

Two views are built from the same raw batch: a **weak** one (random horizontal flip, then
random crop with 4 pixels of padding) and a **strong** one (RandAugment with 3 operations at
magnitude 5, followed by the weak one). They are applied on the CPU to the whole batch, just
before it is moved to the GPU.

**The random parameters are drawn once per batch, not once per image.** Given a batch
tensor, torchvision's v2 transforms apply the same flip, the same crop and the same RandAugment
operations to every image in it. Every run of the paper was made this way, for every method
alike. FixMatch, as originally published, draws them per image. Set `PER_SAMPLE_AUGMENT = True`
to do the same: it is slower, since the images are then transformed one by one, and the results
are no longer those of the paper.

In [ ]:
weak_augment = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(),
    v2.RandomCrop(32, padding=4),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])
strong_augment = v2.Compose([v2.RandAugment(num_ops=3, magnitude=5), *weak_augment.transforms])


def augment(transform, x):
    """Turn a uint8 batch from a loader into an augmented, normalised float batch (on the CPU)."""
    x = x.float().div(255)
    if PER_SAMPLE_AUGMENT:
        return torch.stack([transform(image) for image in x])
    return transform(x)


def to_device(x):
    return x.to(DEVICE, non_blocking=True, memory_format=MEMORY_FORMAT)

In [ ]:
# Both views of eight unlabeled images, spread over the set (which is ordered by class). They form
# one batch, so they share their random parameters. fork_rng restores the random stream
# afterwards, so looking does not change the run.
with torch.random.fork_rng(devices=[]):
    raw = unlabeled_set.tensors[0][torch.linspace(0, len(unlabeled_set) - 1, 8).long()]
    views = {"original": raw.float().div(255),
             "weak": augment(weak_augment, raw),
             "strong": augment(strong_augment, raw)}

mean, std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for row, (name, batch) in enumerate(views.items()):
    if name != "original":
        batch = batch * std + mean
    for col in range(8):
        axes[row, col].imshow(batch[col].permute(1, 2, 0).clamp(0, 1))
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
    axes[row, 0].set_ylabel(name)
plt.tight_layout()
plt.show()

## 4. Model, EMA and optimiser

A WideResNet-28-2 (28-4 on CIFAR-100), trained by SGD with Nesterov momentum. The learning rate
follows FixMatch's rescaled cosine over 2^20 iterations, $\eta_k = 0.03 \cos(7\pi k / 16K)$.
Accuracy is always measured on an exponential moving average (EMA) of the weights, held in a
second copy of the network.

`base_model` is the plain network, and `model` the same network, possibly compiled. They share
their weights, and the EMA reads them from `base_model`.

In [ ]:
model = build_model("wideresnet", NUM_CLASSES, WIDEN_FACTOR, depth=28).to(DEVICE, memory_format=MEMORY_FORMAT)
eval_model = build_model("wideresnet", NUM_CLASSES, WIDEN_FACTOR, depth=28).to(DEVICE, memory_format=MEMORY_FORMAT)


class EMA:
    """Exponential moving average of a network's weights."""

    def __init__(self, net, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in net.state_dict().items()}

    @torch.no_grad()
    def update(self, net):
        for k, v in net.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v)   # integer buffers, such as BatchNorm's batch counter

    def copy_to(self, net):
        net.load_state_dict(self.shadow)


ema = EMA(model, EMA_DECAY)
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda k: max(0.0, math.cos(7 * math.pi * k / (16 * TOTAL_STEPS))))

base_model = model
if COMPILE and DEVICE.type == "cuda":
    try:
        import triton  # noqa: F401  (torch.compile needs it)
        model = torch.compile(model, mode="reduce-overhead")
    except ImportError:
        print("triton is not installed: running without torch.compile")
print(f"{sum(p.numel() for p in base_model.parameters()):,} parameters")

## 5. The method

### ResizeMix and the fairness term

`resizemix` pastes a resized centre crop of a shuffled partner into a random box of each
image, and mixes the labels in proportion to the area. `entropy_loss` is FreeMatch's
fairness term. Both are the reference implementation's.

In [ ]:
def rand_bbox_tao(size, tao):
    """A random box whose side is a fraction `tao` of the image."""
    W, H = size[2], size[3]
    cut_w, cut_h = int(W * tao), int(H * tao)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2


def resizemix(img, gt_label, crop_ratio=0.9, scope=(0.1, 0.8), alpha=1.0):
    """Paste a resized centre crop of a shuffled partner into a random box of each image
    (Qin et al., ResizeMix), mixing the labels in proportion to the area."""
    rand_index = torch.randperm(img.size(0), device=img.device)
    img = img.clone()
    img_resize = img[rand_index]
    shuffled_gt = gt_label[rand_index]
    _, _, h, w = img.size()

    tao = np.sqrt(np.random.beta(alpha, alpha))
    tao = scope[0] + tao * (scope[1] - scope[0])
    bbx1, bby1, bbx2, bby2 = rand_bbox_tao(img.size(), tao)

    crop_h, crop_w = int(h * crop_ratio), int(w * crop_ratio)
    start_h, start_w = (h - crop_h) // 2, (w - crop_w) // 2
    img_resize = img_resize[:, :, start_h:start_h + crop_h, start_w:start_w + crop_w]
    img_resize = F.interpolate(img_resize, (bby2 - bby1, bbx2 - bbx1), mode="nearest")

    img[:, :, bby1:bby2, bbx1:bbx2] = img_resize
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (w * h))
    return img, lam * gt_label + (1 - lam) * shuffled_gt


def entropy_loss(mask, logits_s, p_model, label_hist):
    """FreeMatch's fairness term. Minimising it maximises the cross-entropy between the
    rarity-modulated model distribution and the rarity-modulated prediction histogram of the
    masked strong view, which keeps the pseudo-labels from collapsing onto a few classes."""
    logits_s = logits_s[mask]
    if logits_s.shape[0] == 0:
        return logits_s.new_zeros(())
    prob_s = logits_s.softmax(dim=-1)
    hist_s = torch.bincount(prob_s.argmax(dim=-1), minlength=logits_s.shape[1]).to(prob_s.dtype)
    hist_s = hist_s / hist_s.sum()

    mod_p_model = p_model * torch.nan_to_num(1.0 / label_hist, nan=0.0, posinf=0.0, neginf=0.0)
    mod_p_model = mod_p_model / mod_p_model.sum()
    mod_mean_s = prob_s.mean(dim=0) * torch.nan_to_num(1.0 / hist_s, nan=0.0, posinf=0.0, neginf=0.0)
    mod_mean_s = mod_mean_s / mod_mean_s.sum()
    return (mod_p_model * torch.log(mod_mean_s + 1e-12)).sum()

### Warmup, and seeding the adaptive threshold

Before training proper, `WARMUP_STEPS` supervised-only iterations are run on the labeled
batch, then the moving averages of the threshold are seeded from the predictions on about
4,096 unlabeled images.

**These iterations are run before the clock starts.** They count neither in the reported
time, nor in the number of iterations, nor in the FLOPs, and the learning-rate schedule and
the EMA do not advance during them. This is how every RegMixMatch run of the paper was made.

With persistent workers, `iter(unlabeled_loader)` always returns the same iterator, so the
seeding loop below restarts the unlabeled iterator created here, and the training loop
carries on from where the seeding stopped.

In [ ]:
labeled_iter, unlabeled_iter = iter(labeled_loader), iter(unlabeled_loader)

model.train()
for _ in range(WARMUP_STEPS):
    (x_l, y_l), labeled_iter = next_batch(labeled_iter, labeled_loader)
    x_l = augment(weak_augment, x_l).to(DEVICE, non_blocking=True)
    y_l = y_l.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with amp():
        loss = F.cross_entropy(model(x_l), y_l)
    loss.backward()
    optimizer.step()

model.eval()
probs = []
with torch.no_grad():
    for x_u, _, _ in unlabeled_loader:
        x_u = augment(weak_augment, x_u).to(DEVICE, non_blocking=True)
        probs.append(F.softmax(model(x_u).float(), dim=1).cpu())
        if sum(p.shape[0] for p in probs) >= 4096:
            break
probs = torch.cat(probs)
max_probs_seed, pseudo_seed = probs.max(dim=-1)

# The state of the adaptive threshold: global level, per-class modulation, pseudo-label histogram.
fm = {"time_p": max_probs_seed.mean().to(DEVICE),
      "p_model": probs.mean(dim=0).to(DEVICE),
      "label_hist": (torch.bincount(pseudo_seed, minlength=NUM_CLASSES).float() / len(pseudo_seed)).to(DEVICE)}
model.train()

### The self-adaptive threshold

At every iteration the three moving averages absorb the batch (decay 0.999), then each image
gets the threshold `time_p * p_model[c] / max(p_model)`, where `c` is its pseudo-label.

In [ ]:
def freematch_mask(prob_w):
    """Update the threshold's moving averages with this batch, and return the mask it gives,
    with the confidences and pseudo-labels."""
    max_probs, pseudo = prob_w.max(dim=-1)
    fm["time_p"] = fm["time_p"] * 0.999 + max_probs.mean() * 0.001
    fm["p_model"] = fm["p_model"] * 0.999 + prob_w.mean(dim=0) * 0.001
    hist = torch.bincount(pseudo, minlength=NUM_CLASSES).to(fm["p_model"].dtype)
    fm["label_hist"] = fm["label_hist"] * 0.999 + (hist / hist.sum()) * 0.001

    threshold = fm["time_p"] * (fm["p_model"] / fm["p_model"].max())[pseudo]
    if DATASET == "svhn" and SVHN_CLAMP:
        threshold = threshold.clamp(min=0.9, max=0.95)
    return max_probs.ge(threshold), max_probs, pseudo

### ResizeMix on the confident pool

With few labels per class (fewer than 100), the pool is only the confident unlabeled
images; with more, the labeled batch joins it. The loss is summed over the pool and divided
by the unlabeled batch size, so it grows as more images become confident.

In [ ]:
confident_pool_full = (NUM_LABELED / NUM_CLASSES) >= 100   # labeled images join the pool


def resizemix_loss(x_l, y_l, x_u_s, pseudo, valid):
    n_l, n_u = x_l.shape[0], x_u_s.shape[0]
    one_hot_l, one_hot_u = F.one_hot(y_l, NUM_CLASSES).float(), F.one_hot(pseudo, NUM_CLASSES).float()

    if STATIC_SHAPES:
        # Mix the whole batch, then weight out what is not in the pool: fixed shapes.
        weight = torch.cat([torch.full((n_l,), 1.0 if confident_pool_full else 0.0, device=DEVICE),
                            valid.float()])
        mixed_x, mixed_y = resizemix(torch.cat([x_l, x_u_s]), torch.cat([one_hot_l, one_hot_u]), alpha=ALPHA_H)
        per_image = (F.log_softmax(model(mixed_x), dim=-1) * -mixed_y).sum(dim=-1)
        return (per_image * weight).sum() / n_u

    # Slice out the pool, whose size changes at every iteration.
    if confident_pool_full:
        pool_x, pool_y = torch.cat([x_l, x_u_s[valid]]), torch.cat([one_hot_l, one_hot_u[valid]])
    else:
        pool_x, pool_y = x_u_s[valid], one_hot_u[valid]
    mixed_x, mixed_y = resizemix(pool_x, pool_y, alpha=ALPHA_H)
    return (F.log_softmax(model(mixed_x), dim=-1) * -mixed_y).sum(dim=-1).sum() / n_u

### One training step

One forward pass on the labeled batch and both unlabeled views together gives
$\mathcal{L}_s$, the threshold and $\mathcal{L}_u$; a second pass, on the mixed pool, gives
$\mathcal{L}_{mix}$.

In [ ]:
def train_step(x_l, y_l, x_u, idx, step):
    x_l = to_device(augment(weak_augment, x_l))
    y_l = y_l.to(DEVICE, non_blocking=True)
    x_u_w = to_device(augment(weak_augment, x_u))
    x_u_s = to_device(augment(strong_augment, x_u))
    n_l, n_u = x_l.shape[0], x_u_w.shape[0]

    with amp():
        logits = model(torch.cat([x_l, x_u_w, x_u_s]))
        logits_l, logits_w, logits_s = logits[:n_l], logits[n_l:n_l + n_u], logits[n_l + n_u:]
        loss_s = F.cross_entropy(logits_l, y_l)

        with torch.no_grad():
            prob_w = F.softmax(logits_w.float(), dim=-1)
            mask, max_probs, pseudo = freematch_mask(prob_w)
            valid = max_probs >= TAU_M   # the confident pool

        if HARD_LABEL:
            ce = F.cross_entropy(logits_s, pseudo, reduction="none")
        else:
            soft = F.softmax(logits_w.detach() / T, dim=-1)
            ce = -(soft * F.log_softmax(logits_s, dim=-1)).sum(dim=-1)
        loss_u = (ce * mask.float()).mean()

        loss_mix = resizemix_loss(x_l, y_l, x_u_s, pseudo, valid)
        loss_ent = entropy_loss(mask, logits_s, fm["p_model"], fm["label_hist"])
        loss = loss_s + LAMBDA_E * loss_ent + (LAMBDA_U * loss_u + loss_mix)

    return dict(loss=loss, mask=mask, pseudo=pseudo, confidence=max_probs)

## 6. Evaluation and recording

Every `TEST_PERIOD` iterations, the EMA weights are copied into `eval_model` and scored on the
test set (accuracy and macro-F1). The same evaluation records the training loss and the mask
ratio averaged since the previous one, and how the pseudo-labels moved in between: the share
that is correct (`pl_quality`, the quantity of Figure 1), and how many were newly assigned,
corrected or broken. Everything goes into one JSON file, rewritten at each evaluation, in the
format the scripts write.

In [ ]:
METHOD = "regmixmatch"
wf = f"_wf{WIDEN_FACTOR}" if WIDEN_FACTOR != 2 else ""
assert RUN_TAG, "keep a RUN_TAG, so that a run from here never overwrites one of the paper's result files"
RESULT_DIR = REPO_ROOT / "results" / f"{DATASET}-labeled-{NUM_LABELED}-seed-{SEED}"
RESULT_FILE = RESULT_DIR / f"{METHOD}_ema{wf}_{RUN_TAG}_metrics.json"

METRIC_KEYS = ["step", "train_loss", "test_f1", "test_acc", "time_elapsed", "pl_quality",
               "mask_ratio", "corrections", "new_errors", "error_reinforcement",
               "correct_reinforcement", "new_label", "new_correct", "bad_corrections", "topk_acc"]
metrics = {key: [] for key in METRIC_KEYS}

# The latest pseudo-label and confidence of every unlabeled image (-1: never admitted by the mask
# yet), and the same at the previous evaluation.
pseudo_labels = torch.full((len(unlabeled_idx),), -1, dtype=torch.long)
confidences = torch.zeros(len(unlabeled_idx))
last_pseudo_labels, last_confidences = pseudo_labels.clone(), confidences.clone()


@torch.no_grad()
def evaluate(net):
    """Test accuracy and macro-F1."""
    net.eval()
    mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
    predictions = []
    for x, _ in test_loader:
        x = x.to(DEVICE, non_blocking=True).float().div(255).sub(mean).div(std)
        predictions.append(net(x).argmax(dim=1).cpu())
    predictions = torch.cat(predictions).numpy()
    return accuracy_score(y_test.numpy(), predictions), f1_score(y_test.numpy(), predictions, average="macro")


def track_pseudo_labels(idx, mask, pseudo, confidence):
    """Remember the pseudo-label of every image the mask admitted at this iteration."""
    admitted = mask.bool().cpu()
    pseudo_labels[idx] = torch.where(admitted, pseudo.cpu(), pseudo_labels[idx])
    confidences[idx] = torch.where(admitted, confidence.cpu(), confidences[idx])


def pseudo_label_metrics():
    """How the pseudo-labels moved since the previous evaluation."""
    new = (pseudo_labels != last_pseudo_labels) & (last_pseudo_labels == -1)
    changed = (pseudo_labels != last_pseudo_labels) & (last_pseudo_labels != -1)
    was_good = last_pseudo_labels == unlabeled_true_labels
    is_good = pseudo_labels == unlabeled_true_labels
    more_confident = confidences > last_confidences
    return {
        "pl_quality": is_good.sum().item() / len(unlabeled_true_labels),
        "corrections": (changed & ~was_good & is_good).sum().item(),
        "bad_corrections": (changed & was_good & ~is_good).sum().item(),
        "new_label": new.sum().item(),
        "new_errors": (new & ~is_good).sum().item(),
        "new_correct": (new & is_good).sum().item(),
        "correct_reinforcement": (more_confident & is_good & was_good & ~new).sum().item(),
        "error_reinforcement": (more_confident & ~is_good & ~was_good & ~new).sum().item(),
    }


def evaluate_and_record(step, losses, mask_ratios, start_time):
    """Evaluate the EMA model, append one entry to every metric and rewrite the JSON file.
    Returns True when the run should stop: target accuracy reached, or time budget spent."""
    ema.copy_to(eval_model)
    acc, f1 = evaluate(eval_model)
    elapsed = time.time() - start_time
    entry = {"step": step + 1, "train_loss": float(np.mean(losses)), "test_f1": float(f1),
             "test_acc": float(acc), "time_elapsed": elapsed, "mask_ratio": float(np.mean(mask_ratios)),
             "topk_acc": [float(acc)], **pseudo_label_metrics()}
    for key, value in entry.items():
        metrics[key].append(value)
    last_pseudo_labels.copy_(pseudo_labels)
    last_confidences.copy_(confidences)
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FILE.write_text(json.dumps(metrics, indent=4))

    print(f"step {step + 1:>7} | acc {acc:.4f} | F1 {f1:.4f} | PL quality {entry['pl_quality']:.4f} | "
          f"mask ratio {entry['mask_ratio']:.4f} | loss {entry['train_loss']:.4f} | {elapsed / 60:.1f} min")
    if acc >= TARGET_ACC:
        print(f"Target accuracy {TARGET_ACC:.0%} reached at iteration {step + 1}.")
        return True
    if elapsed >= MAX_MINUTES * 60:
        print(f"Time budget of {MAX_MINUTES} minutes spent at iteration {step + 1}.")
        return True
    return False

## 7. Training

The loop is the same for every method: draw a labeled and an unlabeled batch, compute the
method's loss with `train_step`, take one optimiser step, update the EMA, and evaluate every
`TEST_PERIOD` iterations. The clock starts here, so data loading and model building are not
counted, as in the scripts. The warmup of section 5 has already run, before the clock starts. You can interrupt the cell at any time: the metrics recorded so
far are already on disk.

In [ ]:
losses, mask_ratios = [], []
start_time = time.time()
# labeled_iter and unlabeled_iter were created before the warmup (section 5).

for step in range(MAX_STEPS):
    model.train()
    (x_l, y_l), labeled_iter = next_batch(labeled_iter, labeled_loader)
    (x_u, _, idx), unlabeled_iter = next_batch(unlabeled_iter, unlabeled_loader)

    optimizer.zero_grad(set_to_none=True)
    out = train_step(x_l, y_l, x_u, idx, step)
    out["loss"].backward()
    optimizer.step()
    scheduler.step()
    ema.update(base_model)

    losses.append(out["loss"].item())
    mask_ratios.append(out["mask"].float().mean().item())
    track_pseudo_labels(idx, out["mask"], out["pseudo"], out["confidence"])

    if (step + 1) % TEST_PERIOD == 0 or step == 0 or step == MAX_STEPS - 1:
        stop = evaluate_and_record(step, losses, mask_ratios, start_time)
        losses, mask_ratios = [], []
        if stop:
            break

## 8. Results

This run against the paper's run of the same method, configuration and seed, when that file
exists. The x-axis is iterations: unlike wall-clock time, they do not depend on the machine.

In [ ]:
runs = {"this run": metrics}
paper_file = RESULT_DIR / f"{METHOD}_ema{wf}_metrics.json"
if paper_file.exists():
    runs["paper"] = json.loads(paper_file.read_text())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for label, run in runs.items():
    axes[0].plot(run["step"], run["test_acc"], label=label)
    axes[1].plot(run["step"], run["pl_quality"], label=label)
axes[0].axhline(TARGET_ACC, color="black", linestyle="--", linewidth=1, label=f"target {TARGET_ACC:.0%}")
for ax, title in zip(axes, ["Test accuracy (EMA)", "Pseudo-label quality"]):
    ax.set_xlim(0, metrics["step"][-1])
    ax.set_xlabel("Iterations")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 9. Things to try

- **`TAU_M`**: how confident an image must be to join the pool; lower it and the pool fills
  much earlier.
- **`LAMBDA_E = 0`**: without the fairness term.
- **`WARMUP_STEPS`**: how much the threshold depends on its seeding.
- **`SVHN_CLAMP = False`** on SVHN: the configuration of Appendix A.5, which never reaches 90%.
- **`STATIC_SHAPES = True`**: the masked path, which also turns on `torch.compile`.